# Employee Attrition Prediction

## Project Purpose

This project analyses employee attrition patterns and builds a practical machine learning workflow to help HR identify employees who may be at higher risk of leaving the organisation.

The project focuses on:

- understanding attrition patterns through exploratory data analysis
- preparing clean features for modelling
- handling class imbalance appropriately
- comparing machine learning models using suitable evaluation metrics
- interpreting important predictors to support HR decision-making


In [1]:

# 1. Import libraries

import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

In [48]:
# 2. Create output folders
# Save figures and model comparison outputs for GitHub documentation.

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

In [49]:

# 3. Load dataset


df = pd.read_csv("HR-Employee-Attrition-A2.csv")

display(df.head())

print("Dataset shape:", df.shape)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2.0,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1.0,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2.0,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4.0,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1.0,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


Dataset shape: (1470, 35)


In [33]:

# 4. Initial data inspection

print("Column information:")
df.info()

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isna().sum())

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1470 non-null   int64  
 1   Attrition                 1470 non-null   object 
 2   BusinessTravel            1470 non-null   object 
 3   DailyRate                 1470 non-null   int64  
 4   Department                1470 non-null   object 
 5   DistanceFromHome          1470 non-null   int64  
 6   Education                 1463 non-null   float64
 7   EducationField            1470 non-null   object 
 8   EmployeeCount             1470 non-null   int64  
 9   EmployeeNumber            1470 non-null   int64  
 10  EnvironmentSatisfaction   1470 non-null   int64  
 11  Gender                    1470 non-null   object 
 12  HourlyRate                1470 non-null   int64  
 13  JobInvolvement            1470 non-null   i

## Target Variable Analysis

The target variable is `Attrition`, which indicates whether an employee left the organisation.

Before modelling, it is important to check the class distribution because employee attrition is often an imbalanced classification problem. If most employees did not leave, accuracy alone can give a misleading view of model performance.

In [34]:
# ============================================================
# 5. Target variable analysis
# ============================================================

# Purpose:
# Check class imbalance. This is important because attrition is usually rare.

attrition_counts = df["Attrition"].value_counts()
attrition_rate = df["Attrition"].value_counts(normalize=True) * 100

print("Attrition counts:")
print(attrition_counts)

print("\nAttrition rate (%):")
print(attrition_rate.round(2))

fig = px.bar(
    x=attrition_counts.index,
    y=attrition_counts.values,
    labels={"x": "Attrition", "y": "Number of Employees"},
    title="Employee Attrition Distribution"
)

fig.show()
fig.write_html("outputs/figures/attrition_distribution.html")

Attrition counts:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition rate (%):
Attrition
No     83.88
Yes    16.12
Name: proportion, dtype: float64


# Exploratory Data Analysis

This section explores employee attrition patterns across key business variables. For categorical variables, both employee counts and attrition rates are used so that group size does not lead to misleading conclusions.

In [66]:

# 6. Data cleaning copy for EDA

# Purpose:
# Keep original dataset unchanged.
# For EDA only, fill missing Education values using the mode.

df_eda = df.copy()
df_eda["Education"] = df_eda["Education"].fillna(df_eda["Education"].mode()[0]).astype(int)

df_eda["Attrition_encoded"] = df_eda["Attrition"].map({"No": 0, "Yes": 1})

In [9]:

# 7. Business-focused EDA


# Purpose:
# Explore practical HR factors related to attrition.


fig = px.histogram(
    df_eda,
    x="OverTime",
    color="Attrition",
    barmode="group",
    title="Attrition by Overtime"
)

fig.show()
fig.write_html("outputs/figures/attrition_by_overtime.html")



In [67]:
overtime_rate = (
    df_eda.groupby("OverTime")["Attrition_encoded"]
    .mean()
    .mul(100)
    .reset_index(name="Attrition Rate (%)")
)

display(overtime_rate.round(1))

fig = px.bar(
    overtime_rate,
    x="OverTime",
    y="Attrition Rate (%)",
    title="Attrition Rate by Overtime",
    text="Attrition Rate (%)"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()
fig.write_html("outputs/figures/attrition_rate_by_overtime.html")

,OverTime,Attrition Rate (%)
0,No,10.4
1,Yes,30.5


### Interpretation

The attrition rate differs substantially between employees who work overtime and those who do not.

Approximately **30.5%** of employees who work overtime left the organisation, compared with **10.4%** of employees who do not work overtime.

This suggests that employees who work overtime were represented more frequently among employees who left the organisation in this dataset.

Although this finding indicates an association between overtime and attrition, it should not be interpreted as evidence that overtime directly causes employees to leave. Additional factors may also contribute to employee turnover.

In [68]:


fig = px.histogram(
    df_eda,
    x="Department",
    color="Attrition",
    barmode="group",
    title="Attrition by Department"
)

fig.show()
fig.write_html("outputs/figures/attrition_by_department.html")





department_rate = (
    df_eda.groupby("Department")["Attrition_encoded"]
    .mean()
    .mul(100)
    .reset_index(name="Attrition Rate (%)")
    .sort_values("Attrition Rate (%)", ascending=False)
)

display(department_rate.round(1))

fig = px.bar(
    department_rate,
    x="Department",
    y="Attrition Rate (%)",
    title="Attrition Rate by Department",
    text="Attrition Rate (%)"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()
fig.write_html("outputs/figures/attrition_rate_by_department.html")

,Department,Attrition Rate (%)
2,Sales,20.6
0,Human Resources,19.0
1,Research & Development,13.8


### Interpretation

The attrition rate varies across departments within the organisation.

Employees in the **Sales** department experienced the highest attrition rate (**20.6%**), followed by **Human Resources** (**19.0%**). In comparison, **Research & Development** recorded the lowest attrition rate (**13.8%**).

These results suggest that employee turnover was more common in Sales and Human Resources than in Research & Development within this dataset.

Although department-level differences are evident, these results describe associations observed in the data and should not be interpreted as evidence that department membership alone causes employee attrition.

In [69]:


fig = px.histogram(
    df_eda,
    x="JobRole",
    color="Attrition",
    barmode="group",
    title="Attrition by Job Role"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()
fig.write_html("outputs/figures/attrition_by_jobrole.html")




jobrole_rate = (
    df_eda.groupby("JobRole")["Attrition_encoded"]
    .mean()
    .mul(100)
    .reset_index(name="Attrition Rate (%)")
    .sort_values("Attrition Rate (%)", ascending=False)
)

display(jobrole_rate.round(1))

fig = px.bar(
    jobrole_rate,
    x="JobRole",
    y="Attrition Rate (%)",
    title="Attrition Rate by Job Role",
    text="Attrition Rate (%)"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.update_layout(xaxis_tickangle=-45)

fig.show()

fig.write_html("outputs/figures/attrition_rate_by_jobrole.html")

,JobRole,Attrition Rate (%)
8,Sales Representative,39.8
2,Laboratory Technician,23.9
1,Human Resources,23.1
7,Sales Executive,17.5
6,Research Scientist,16.1
4,Manufacturing Director,6.9
0,Healthcare Representative,6.9
3,Manager,4.9
5,Research Director,2.5


### Interpretation

Attrition rates vary considerably across job roles within the organisation.

The **Sales Representative** role recorded the highest attrition rate (**39.8%**), followed by **Laboratory Technician** (**23.9%**) and **Human Resources** (**23.1%**). In contrast, **Research Director** had the lowest attrition rate (**2.5%**), with **Manager** (**4.9%**) and **Healthcare Representative** (**6.9%**) also showing relatively low attrition rates.

These findings suggest that employee turnover was more common in certain job roles than others within this dataset. Such differences may help HR identify roles that warrant further investigation and targeted retention strategies.

These results describe associations observed in the dataset and should not be interpreted as evidence that a particular job role directly causes employee attrition.

In [43]:


fig = px.box(
    df_eda,
    x="Attrition",
    y="MonthlyIncome",
    title="Monthly Income by Attrition",
    category_orders={"Attrition": ["No", "Yes"]}
)

fig.show()
fig.write_html("outputs/figures/monthly_income_by_attrition.html")

### Interpretation

The distribution of monthly income differs between employees who left the organisation and those who remained.

Employees who experienced attrition generally had a **lower median monthly income** than employees who stayed. In addition, employees who remained with the organisation exhibited a **wider distribution of monthly income**, including a greater number of high-income observations.

These findings suggest that lower monthly income was more commonly observed among employees who left the organisation in this dataset. However, this chart describes an association and should not be interpreted as evidence that income alone determines employee attrition.

In [13]:


fig = px.box(
    df_eda,
    x="Attrition",
    y="YearsAtCompany",
    title="Years at Company by Attrition"
)

fig.show()
fig.write_html("outputs/figures/years_at_company_by_attrition.html")

### Interpretation

The distribution of years at the company differs between employees who left the organisation and those who remained.

Employees who experienced attrition generally had a **shorter tenure**, with a lower median number of years at the company compared with employees who stayed. Employees who remained with the organisation also exhibited a wider range of tenure, including many long-serving employees.

These findings suggest that shorter organisational tenure was more commonly observed among employees who left the organisation in this dataset. However, this figure describes an association and should not be interpreted as evidence that tenure alone determines employee attrition.

In [46]:


fig = px.histogram(
    df_eda,
    x="JobSatisfaction",
    color="Attrition",
    barmode="group",
    title="Attrition by Job Satisfaction",
    category_orders={"Attrition": ["No", "Yes"]}
)

fig.show()
fig.write_html("outputs/figures/attrition_by_job_satisfaction.html")

### Interpretation

This figure shows the number of employees with and without attrition across different job satisfaction levels.

Because the number of employees differs across satisfaction levels, employee counts alone do not provide a fair comparison of attrition risk.

To compare the likelihood of attrition across satisfaction levels, the attrition rate within each satisfaction category is examined in the following figure.

In [44]:




jobsat_rate = (
    df_eda.groupby("JobSatisfaction")["Attrition_encoded"]
    .mean()
    .mul(100)
    .reset_index(name="Attrition Rate (%)")
    .sort_values("JobSatisfaction")
)

display(jobsat_rate.round(1))

fig = px.bar(
    jobsat_rate,
    x="JobSatisfaction",
    y="Attrition Rate (%)",
    title="Attrition Rate by Job Satisfaction",
    text="Attrition Rate (%)"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

fig.write_html("outputs/figures/attrition_rate_by_jobsatisfaction.html")

,JobSatisfaction,Attrition Rate (%)
0,1,22.8
1,2,16.4
2,3,16.5
3,4,11.3


### Interpretation

The attrition rate varies across job satisfaction levels.

Employees with the **lowest job satisfaction (Level 1)** experienced the highest attrition rate (**22.8%**), whereas employees with the **highest job satisfaction (Level 4)** had the lowest attrition rate (**11.3%**). Attrition rates for satisfaction levels **2** and **3** were similar, at approximately **16.4%** and **16.5%**, respectively.

These findings suggest that lower job satisfaction was associated with a higher proportion of employees leaving the organisation in this dataset. However, this analysis identifies an association and should not be interpreted as evidence that job satisfaction alone causes employee attrition.

In [15]:

# 8. Attrition percentage tables


# Purpose:
# Percentages are more useful than raw counts for business interpretation.

def attrition_rate_table(data, column):
    table = (
        data.groupby(column)["Attrition_encoded"]
        .agg(["count", "mean"])
        .reset_index()
        .rename(columns={"count": "Employee_Count", "mean": "Attrition_Rate"})
    )
    table["Attrition_Rate"] = table["Attrition_Rate"] * 100
    return table.sort_values("Attrition_Rate", ascending=False)

for col in ["OverTime", "Department", "JobRole", "JobSatisfaction", "WorkLifeBalance"]:
    print(f"\nAttrition rate by {col}:")
    display(attrition_rate_table(df_eda, col))


Attrition rate by OverTime:


,OverTime,Employee_Count,Attrition_Rate
1,Yes,416,30.528846
0,No,1054,10.436433



Attrition rate by Department:


,Department,Employee_Count,Attrition_Rate
2,Sales,446,20.627803
0,Human Resources,63,19.047619
1,Research & Development,961,13.839750



Attrition rate by JobRole:


,JobRole,Employee_Count,Attrition_Rate
8,Sales Representative,83,39.759036
2,Laboratory Technician,259,23.938224
1,Human Resources,52,23.076923
7,Sales Executive,326,17.484663
6,Research Scientist,292,16.095890
4,Manufacturing Director,145,6.896552
0,Healthcare Representative,131,6.870229
3,Manager,102,4.901961
5,Research Director,80,2.500000



Attrition rate by JobSatisfaction:


,JobSatisfaction,Employee_Count,Attrition_Rate
0,1,289,22.837370
2,3,442,16.515837
1,2,280,16.428571
3,4,459,11.328976



Attrition rate by WorkLifeBalance:


,WorkLifeBalance,Employee_Count,Attrition_Rate
0,1,80,31.250000
3,4,153,17.647059
1,2,344,16.860465
2,3,893,14.221725


In [16]:

# 9. Correlation analysis
# Identify numeric variables with linear association to attrition.

numeric_corr = df_eda.select_dtypes(include=["int64", "float64"]).corr()

attrition_corr = (
    numeric_corr["Attrition_encoded"]
    .drop("Attrition_encoded")
    .sort_values(ascending=False)
)

print("Correlation with Attrition:")
display(attrition_corr)

fig = px.imshow(
    numeric_corr,
    title="Correlation Heatmap of Numeric Variables",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)

fig.show()
fig.write_html("outputs/figures/correlation_heatmap.html")

Correlation with Attrition:


,Attrition_encoded
DistanceFromHome,0.077924
NumCompaniesWorked,0.043494
MonthlyRate,0.015170
PerformanceRating,0.002889
HourlyRate,-0.006846
EmployeeNumber,-0.010577
PercentSalaryHike,-0.013478
Education,-0.030814
YearsSinceLastPromotion,-0.033019
RelationshipSatisfaction,-0.045872


# Feature Engineering

To improve business interpretability, several additional features were derived from existing variables. Each engineered feature was created using only information already available in the original dataset and was designed to capture meaningful HR concepts without introducing data leakage.

In [17]:

# 10. Feature engineering


# Create business-driven features derived only from existing columns.
# These features are designed to be explainable to HR stakeholders.
df_model = df.copy()

# Remove non-informative columns:
# EmployeeCount, Over18, and StandardHours are constants.
# EmployeeNumber is an identifier.

columns_to_drop = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
]

df_model = df_model.drop(columns=columns_to_drop)

# Feature 1: Career stage based on TotalWorkingYears

df_model["CareerStage"] = pd.cut(
    df_model["TotalWorkingYears"],
    bins=[-1, 5, 15, 100],
    labels=["Early Career", "Mid Career", "Senior Career"]
)

# Feature 2: Long commute indicator based on DistanceFromHome

df_model["CommuteDistanceGroup"] = pd.cut(
    df_model["DistanceFromHome"],
    bins=[-1, 5, 15, 100],
    labels=["Near", "Medium", "Far"]
)

# Feature 3: Promotion delay ratio
# Avoid division by zero by replacing zero YearsAtCompany with 1.

df_model["PromotionDelayRatio"] = (
    df_model["YearsSinceLastPromotion"] / df_model["YearsAtCompany"].replace(0, 1)
)

# Feature 4: Manager stability ratio

df_model["ManagerStabilityRatio"] = (
    df_model["YearsWithCurrManager"] / df_model["YearsAtCompany"].replace(0, 1)
)

# Feature 5: Income per working year
# Avoid division by zero by replacing zero TotalWorkingYears with 1.

df_model["IncomePerWorkingYear"] = (
    df_model["MonthlyIncome"] / df_model["TotalWorkingYears"].replace(0, 1)
)

display(df_model.head())

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,CareerStage,CommuteDistanceGroup,PromotionDelayRatio,ManagerStabilityRatio,IncomePerWorkingYear
0,41,Yes,Travel_Rarely,1102,Sales,1,2.0,Life Sciences,2,Female,...,1,6,4,0,5,Mid Career,Near,0.000,0.833333,749.125000
1,49,No,Travel_Frequently,279,Research & Development,8,1.0,Life Sciences,3,Male,...,3,10,7,1,7,Mid Career,Medium,0.100,0.700000,513.000000
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2.0,Other,4,Male,...,3,0,0,0,0,Mid Career,Near,0.000,0.000000,298.571429
3,33,No,Travel_Frequently,1392,Research & Development,3,4.0,Life Sciences,4,Female,...,3,8,7,3,0,Mid Career,Near,0.375,0.000000,363.625000
4,27,No,Travel_Rarely,591,Research & Development,2,1.0,Medical,1,Male,...,3,2,2,2,2,Mid Career,Near,1.000,1.000000,578.000000


In [18]:

# 11. Prepare target and features


# Purpose:
# Prevent data leakage by removing Attrition from the feature matrix.

y = df_model["Attrition"].map({"No": 0, "Yes": 1})
X = df_model.drop(columns=["Attrition"])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (1470, 35)
Target shape: (1470,)


In [19]:

# 12. Identify feature types


numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'PromotionDelayRatio', 'ManagerStabilityRatio', 'IncomePerWorkingYear']

Categorical features:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'CareerStage', 'CommuteDistanceGroup']


In [20]:

# 13. Train-test split


# Purpose:
# Use stratification because only around 16% of employees have Attrition = Yes.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (1176, 35)
Testing set: (294, 35)

Training target distribution:
Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64

Testing target distribution:
Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


# Data Preprocessing

The dataset contains both numerical and categorical variables.

A Scikit-learn preprocessing pipeline was used to:

- impute missing numerical values using the median
- standardise numerical variables
- one-hot encode categorical variables

This preprocessing was applied consistently during model training using Scikit-learn pipelines to prevent data leakage.

In [63]:

# 14. Preprocessing pipeline

# Numeric variables:
# - Impute missing values using median
# - Standardise values
#
# Categorical variables:
# - Impute missing values using most frequent category
# - One-hot encode categories

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

In [64]:

# 15. Define models


# Purpose:
# Compare a small, realistic set of models:
# - Logistic Regression: interpretable baseline
# - Random Forest: strong and interpretable tree-based model
# - SVM: strong predictive model, but less interpretable
#
# class_weight="balanced" is used because the dataset is imbalanced.

models = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=10000, class_weight="balanced"),
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10]
        }
    },

    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, class_weight="balanced"),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [5, 10, None],
            "classifier__min_samples_split": [2, 5]
        }
    },

    "Support Vector Machine": {
        "model": SVC(probability=True, class_weight="balanced"),
        "params": {
            "classifier__C": [0.1, 1, 10],
            "classifier__kernel": ["linear", "rbf"]
        }
    }
}

## Baseline Model

Before evaluating machine learning models, a simple baseline model is created using `DummyClassifier`.

This baseline predicts the majority class and helps confirm whether the machine learning models perform better than a naive approach.

In [65]:
from sklearn.dummy import DummyClassifier

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent"))
    ]
)

baseline_pipeline.fit(X_train, y_train)

y_pred_baseline = baseline_pipeline.predict(X_test)
y_prob_baseline = baseline_pipeline.predict_proba(X_test)[:, 1]

baseline_results = {
    "Model": "Baseline Dummy Classifier",
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(y_test, y_pred_baseline, zero_division=0),
    "Recall": recall_score(y_test, y_pred_baseline, zero_division=0),
    "F1_Score": f1_score(y_test, y_pred_baseline, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob_baseline),
    "Average_Precision": average_precision_score(y_test, y_prob_baseline)
}

baseline_results

{'Model': 'Baseline Dummy Classifier',
 'Accuracy': 0.8401360544217688,
 'Precision': 0.0,
 'Recall': 0.0,
 'F1_Score': 0.0,
 'ROC_AUC': np.float64(0.5),
 'Average_Precision': np.float64(0.1598639455782313)}

## Baseline Model Interpretation

The Dummy Classifier predicts the majority class (`No Attrition`) for every employee.

Although it achieves an accuracy of approximately 84%, it completely fails to identify employees who actually leave the organisation, resulting in zero precision, recall, and F1-score for the attrition class.

This demonstrates that accuracy alone is not an appropriate evaluation metric for this project. Instead, metrics such as precision, recall, F1-score, ROC-AUC, and Average Precision provide a more meaningful assessment of model performance on this imbalanced classification problem.

The machine learning models developed later in this notebook are therefore expected to outperform this naive baseline across these evaluation metrics.

# Model Evaluation

Three supervised machine learning algorithms were evaluated using the same preprocessing pipeline and hyperparameter tuning process.

Because employee attrition is an imbalanced classification problem (approximately 16% positive cases), model performance was assessed using multiple evaluation metrics rather than accuracy alone.

The primary evaluation metric for model selection was the **F1-score**, as it provides a balanced measure of precision and recall for identifying employees at risk of attrition.

Additional metrics, including ROC-AUC and Average Precision, were also considered to provide a more comprehensive assessment of classification performance.

In [59]:

# 16. Train and evaluate models


# Purpose:
# Optimise using F1-score because attrition is imbalanced.
# Accuracy alone is not enough for this business problem.

results = []
best_models = {}

for model_name, model_info in models.items():

    print("=" * 70)
    print(f"Training model: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model_info["model"])
        ]
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=model_info["params"],
        cv=5,
        scoring="f1",
        n_jobs=-1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_models[model_name] = best_model

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)

    results.append({
        "Model": model_name,
        "Best_CV_F1": grid_search.best_score_,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc,
        "Average_Precision": avg_precision,
        "Best_Parameters": grid_search.best_params_
    })

    print("Best parameters:", grid_search.best_params_)
    print("Best CV F1:", round(grid_search.best_score_, 4))
    print("Test Accuracy:", round(accuracy, 4))
    print("Test Precision:", round(precision, 4))
    print("Test Recall:", round(recall, 4))
    print("Test F1:", round(f1, 4))
    print("Test ROC AUC:", round(roc_auc, 4))
    print("Test Average Precision:", round(avg_precision, 4))

Training model: Logistic Regression
Best parameters: {'classifier__C': 0.1}
Best CV F1: 0.5064
Test Accuracy: 0.7755
Test Precision: 0.3827
Test Recall: 0.6596
Test F1: 0.4844
Test ROC AUC: 0.8125
Test Average Precision: 0.5803
Training model: Random Forest
Best parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV F1: 0.4929
Test Accuracy: 0.8333
Test Precision: 0.48
Test Recall: 0.5106
Test F1: 0.4948
Test ROC AUC: 0.7789
Test Average Precision: 0.4081
Training model: Support Vector Machine
Best parameters: {'classifier__C': 1, 'classifier__kernel': 'rbf'}
Best CV F1: 0.5588
Test Accuracy: 0.8265
Test Precision: 0.4667
Test Recall: 0.5957
Test F1: 0.5234
Test ROC AUC: 0.8008
Test Average Precision: 0.532


In [60]:

# 17. Model comparison table


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1_Score", ascending=False)

display(results_df)

results_df.to_csv("outputs/tables/model_comparison.csv", index=False)

,Model,Best_CV_F1,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Average_Precision,Best_Parameters
2,Support Vector Machine,0.558763,0.826531,0.466667,0.595745,0.523364,0.800758,0.531966,"{'classifier__C': 1, 'classifier__kernel': 'rbf'}"
1,Random Forest,0.492851,0.833333,0.480000,0.510638,0.494845,0.778878,0.408110,"{'classifier__max_depth': 5, 'classifier__min_..."
0,Logistic Regression,0.506403,0.775510,0.382716,0.659574,0.484375,0.812473,0.580317,{'classifier__C': 0.1}


In [61]:

# 18. Visualise model comparison


fig = px.bar(
    results_df,
    x="Model",
    y=["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC", "Average_Precision"],
    barmode="group",
    title="Model Performance Comparison"
)

fig.show()
fig.write_html("outputs/figures/model_performance_comparison.html")

In [26]:

# 19. Select best predictive model


# Purpose:
# Select best model based on test F1-score.
# This balances precision and recall for the attrition class.

best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("Best predictive model based on F1-score:", best_model_name)
display(results_df.iloc[0])

Best predictive model based on F1-score: Support Vector Machine


,2
Model,Support Vector Machine
Best_CV_F1,0.558763
Accuracy,0.826531
Precision,0.466667
Recall,0.595745
F1_Score,0.523364
ROC_AUC,0.800586
Average_Precision,0.531919
Best_Parameters,"{'classifier__C': 1, 'classifier__kernel': 'rbf'}"


## Final Model Selection

Among the evaluated models, the **Support Vector Machine (SVM)** achieved the highest F1-score and was selected as the final predictive model.

Although Logistic Regression achieved strong ROC-AUC and recall performance, the SVM provided the best overall balance between identifying employees at risk of attrition and limiting false predictions.

Random Forest was also retained because it provides feature importance, allowing the model to support business interpretation even though it was not selected as the final predictive model.

In [27]:

# 20. Detailed evaluation of best model


y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=["No Attrition", "Attrition"]))

cm = confusion_matrix(y_test, y_pred_best)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Attrition", "Actual Attrition"],
    columns=["Predicted No Attrition", "Predicted Attrition"]
)

display(cm_df)

fig = px.imshow(
    cm_df,
    text_auto=True,
    title=f"Confusion Matrix - {best_model_name}",
    color_continuous_scale="Blues"
)

fig.show()
fig.write_html("outputs/figures/confusion_matrix.html")

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.92      0.87      0.89       247
   Attrition       0.47      0.60      0.52        47

    accuracy                           0.83       294
   macro avg       0.69      0.73      0.71       294
weighted avg       0.85      0.83      0.83       294



,Predicted No Attrition,Predicted Attrition
Actual No Attrition,215,32
Actual Attrition,19,28


## Classification Report Interpretation

The final Support Vector Machine correctly classified most employees who remained with the organisation while identifying a substantial proportion of employees who left.

As expected for an imbalanced classification problem, predicting employee attrition is more challenging than predicting employees who remain.

The model should therefore be viewed as a decision-support tool that helps HR identify employees who may require further attention rather than as a replacement for managerial decision-making.

In [28]:

# 21. ROC curve


# Purpose:
# Show the model's ability to separate attrition and non-attrition cases.

fpr, tpr, _ = roc_curve(y_test, y_prob_best)

roc_df = pd.DataFrame({
    "False Positive Rate": fpr,
    "True Positive Rate": tpr
})

fig = px.line(
    roc_df,
    x="False Positive Rate",
    y="True Positive Rate",
    title=f"ROC Curve - {best_model_name}"
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(dash="dash")
)

fig.show()
fig.write_html("outputs/figures/roc_curve.html")

### ROC Curve Interpretation

The ROC curve illustrates the trade-off between the true positive rate and false positive rate across different classification thresholds.

The Support Vector Machine achieved a strong ROC-AUC score, indicating good overall discrimination between employees who remained and employees who left.

In [29]:

# 22. Precision-Recall curve
# Precision-Recall curve is useful because attrition is imbalanced.

precision, recall, _ = precision_recall_curve(y_test, y_prob_best)

pr_df = pd.DataFrame({
    "Precision": precision,
    "Recall": recall
})

fig = px.line(
    pr_df,
    x="Recall",
    y="Precision",
    title=f"Precision-Recall Curve - {best_model_name}"
)

fig.show()
fig.write_html("outputs/figures/precision_recall_curve.html")

### Precision–Recall Curve Interpretation

Because employee attrition is an imbalanced classification problem, the Precision–Recall curve provides additional insight into model performance.

The curve demonstrates the model's ability to identify employees at risk of attrition while balancing the trade-off between precision and recall.

This complements the ROC analysis and provides a more informative evaluation for the minority class.

In [30]:
rf_model = best_models["Random Forest"]

rf_classifier = rf_model.named_steps["classifier"]
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_classifier.feature_importances_
})

importance_df = importance_df.sort_values(by="Importance", ascending=False)

display(importance_df.head(20))

importance_df.to_csv("outputs/tables/random_forest_feature_importance.csv", index=False)

fig = px.bar(
    importance_df.head(20).sort_values(by="Importance"),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 20 Feature Importances - Random Forest Interpretation Model"
)

fig.show()
fig.write_html("outputs/figures/random_forest_feature_importance.html")

,Feature,Importance
9,numeric__MonthlyIncome,0.074429
16,numeric__TotalWorkingYears,0.067767
0,numeric__Age,0.063644
19,numeric__YearsAtCompany,0.051616
52,categorical__OverTime_No,0.051348
25,numeric__IncomePerWorkingYear,0.048925
53,categorical__OverTime_Yes,0.047909
24,numeric__ManagerStabilityRatio,0.047626
15,numeric__StockOptionLevel,0.043248
22,numeric__YearsWithCurrManager,0.040580


## Feature Importance Interpretation

The Random Forest model was used to estimate the relative importance of predictor variables.

Feature importance indicates which variables contributed most to the model's predictions, but it does not imply that these variables directly cause employee attrition.

These results provide HR practitioners with useful insights into which employee characteristics warrant further investigation and may support future workforce planning initiatives.

In [62]:

# 23. Business summary based on actual model outputs
# Produce a truthful summary using calculated results only.

best_row = results_df.iloc[0]

print("=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

print(f"Best predictive model: {best_model_name}")
print(f"Accuracy: {best_row['Accuracy']:.4f}")
print(f"Precision: {best_row['Precision']:.4f}")
print(f"Recall: {best_row['Recall']:.4f}")
print(f"F1 Score: {best_row['F1_Score']:.4f}")
print(f"ROC AUC: {best_row['ROC_AUC']:.4f}")
print(f"Average Precision: {best_row['Average_Precision']:.4f}")

print("\nInterpretation note:")
print(
    "Because employee attrition is imbalanced, this project does not rely only on accuracy. "
    "F1-score, recall, ROC-AUC, and average precision are also used to evaluate model performance."
)

print("\nModel interpretation:")
print(
    "Random Forest feature importance is used to identify influential predictors. "
    "This supports business interpretation, especially if the best predictive model is less interpretable."
)

PROJECT SUMMARY
Best predictive model: Support Vector Machine
Accuracy: 0.8265
Precision: 0.4667
Recall: 0.5957
F1 Score: 0.5234
ROC AUC: 0.8008
Average Precision: 0.5320

Interpretation note:
Because employee attrition is imbalanced, this project does not rely only on accuracy. F1-score, recall, ROC-AUC, and average precision are also used to evaluate model performance.

Model interpretation:
Random Forest feature importance is used to identify influential predictors. This supports business interpretation, especially if the best predictive model is less interpretable.


# Project Conclusion

This project developed an end-to-end machine learning workflow for predicting employee attrition. The workflow demonstrates how data analytics and machine learning can support evidence-based HR decision-making while recognising that predictive models complement, rather than replace, human judgement.

The analysis included exploratory data analysis, business-driven feature engineering, data preprocessing using Scikit-learn pipelines, baseline model comparison, hyperparameter tuning, model evaluation, and feature importance interpretation.

Because the dataset is imbalanced, model performance was evaluated using precision, recall, F1-score, ROC-AUC, and Average Precision rather than accuracy alone.

Among the evaluated models, the Support Vector Machine achieved the strongest overall performance based on the F1-score and was selected as the final predictive model. A Random Forest model was also used to support feature importance analysis and improve business interpretability.

The model should be used as a decision-support tool to help HR identify employees who may be at higher risk of attrition rather than as an automated decision-making system.

## Limitations

This analysis is based on historical employee records from a single dataset. The identified relationships are observational and should not be interpreted as causal. Future work could incorporate additional organisational variables, temporal information, or external datasets to further improve predictive performance and generalisability.